# Training for OpenWakeWord

## Install requirements in a new virtual environment
```bash
python3 -m venv venv
source venv/bin/activate
pip install -U pip
pip install openwakeword datasets scipy torch onnx tqdm
```

## Imports

In [ ]:
import json
import math
import random
from pathlib import Path
import datasets
import numpy as np
import openwakeword
import openwakeword.data
import openwakeword.utils
import scipy.io.wavfile
import torch
from numpy.lib.format import open_memmap
from torch import nn
from tqdm import tqdm

## Configuration

In [ ]:
model_name = "hey_atlas"

positive_dir = Path("./content/heyatlas/pos")  
negative_dir = Path("./content/heyatlas/neg")   
output_dir = Path("./content/output")

sample_rate = 16000
clip_seconds = 3

epochs = 12
layer_dim = 32
train_batch_size = 512

feature_file_batch_size = 64
feature_embed_batch_size = 512
feature_ncpu = 4

snr_low = 5
snr_high = 15

min_positive_secs = 0.3
max_positive_secs = 2.95
min_negative_secs = 1.0
max_negative_secs = 1800.0

validation_ratio = 0.15
seed = 42

## Helper methods

In [ ]:
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


def ensure_dir_exists(path: Path, label: str):
    if not path.exists() or not path.is_dir():
        raise FileNotFoundError(f"{label} does not exist or is not a directory: {path}")


def ensure_wavs_exist(path: Path, label: str):
    count = sum(1 for _ in path.rglob("*.wav"))
    if count == 0:
        raise ValueError(f"No .wav files found in {label}: {path}")


def filter_paths(folder: Path, min_secs: float, max_secs: float):
    clips, durations = openwakeword.data.filter_audio_paths(
        [str(folder)],
        min_length_secs=min_secs,
        max_length_secs=max_secs,
        duration_method="header",
    )
    return list(clips), list(durations)


def split_train_val(X: np.ndarray, y: np.ndarray, seed: int, val_ratio: float = 0.15):
    rng = np.random.default_rng(seed)
    indices = rng.permutation(len(X))
    split = max(1, int(len(X) * (1 - val_ratio)))
    train_idx = indices[:split]
    val_idx = indices[split:]
    if len(val_idx) == 0:
        val_idx = train_idx[:1]
    return X[train_idx], X[val_idx], y[train_idx], y[val_idx]

## Build negative samples

In [ ]:
def build_negative_features(
    feature_extractor,
    negative_paths,
    negative_durations,
    output_path: Path,
    clip_seconds: int,
    file_batch_size: int,
    embed_batch_size: int,
    ncpu: int,
):
    audio_dataset = datasets.Dataset.from_dict({"audio": list(negative_paths)})
    audio_dataset = audio_dataset.cast_column("audio", datasets.Audio(sampling_rate=sample_rate))

    n_rows = max(1, int(sum(negative_durations) // clip_seconds))
    shape = feature_extractor.get_embedding_shape(clip_seconds)

    mmap = open_memmap(
        output_path,
        mode="w+",
        dtype=np.float32,
        shape=(n_rows, shape[0], shape[1]),
    )

    row_counter = 0

    for i in tqdm(range(0, audio_dataset.num_rows, file_batch_size), desc="Embedding negative clips"):
        wav_data = [
            (item["array"] * 32767).astype(np.int16)
            for item in audio_dataset[i : i + file_batch_size]["audio"]
        ]

        stacked = openwakeword.data.stack_clips(
            wav_data,
            clip_size=sample_rate * clip_seconds
        ).astype(np.int16)

        features = feature_extractor.embed_clips(
            x=stacked,
            batch_size=embed_batch_size,
            ncpu=ncpu
        )

        remaining = n_rows - row_counter
        if remaining <= 0:
            break

        to_write = features[:remaining]
        mmap[row_counter : row_counter + to_write.shape[0], :, :] = to_write
        row_counter += to_write.shape[0]
        mmap.flush()

    openwakeword.data.trim_mmap(str(output_path))
    return np.load(output_path)

## Build positive samples

In [ ]:
def build_positive_features(
    feature_extractor,
    positive_paths,
    positive_durations,
    negative_paths,
    output_path: Path,
    clip_seconds: int,
    mix_batch_size: int,
    embed_batch_size: int,
    snr_low: int,
    snr_high: int,
):
    total_length = sample_rate * clip_seconds

    jitters = (np.random.uniform(0, 0.2, len(positive_paths)) * sample_rate).astype(np.int32)
    starts = [
        total_length - (int(math.ceil(d * sample_rate)) + j)
        for d, j in zip(positive_durations, jitters)
    ]

    mixing_generator = openwakeword.data.mix_clips_batch(
        foreground_clips=list(positive_paths),
        background_clips=list(negative_paths),
        combined_size=total_length,
        batch_size=mix_batch_size,
        snr_low=snr_low,
        snr_high=snr_high,
        start_index=starts,
        volume_augmentation=True,
    )

    shape = feature_extractor.get_embedding_shape(clip_seconds)

    mmap = open_memmap(
        output_path,
        mode="w+",
        dtype=np.float32,
        shape=(len(positive_paths), shape[0], shape[1]),
    )

    row_counter = 0
    total_batches = max(1, math.ceil(len(positive_paths) / mix_batch_size))

    for batch in tqdm(mixing_generator, total=total_batches, desc="Embedding positive clips"):
        mixed_audio = batch[0]
        features = feature_extractor.embed_clips(
            mixed_audio,
            batch_size=embed_batch_size
        )

        end = min(row_counter + features.shape[0], len(positive_paths))
        mmap[row_counter:end, :, :] = features[: end - row_counter]
        row_counter = end
        mmap.flush()

        if row_counter >= len(positive_paths):
            break

    openwakeword.data.trim_mmap(str(output_path))
    return np.load(output_path)

## Configure model training

In [ ]:
class SimpleWakeWordNet(nn.Module):
    def __init__(self, time_steps: int, feature_dim: int, layer_dim: int):
        super().__init__()
        input_dim = time_steps * feature_dim
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(input_dim, layer_dim),
            nn.LayerNorm(layer_dim),
            nn.ReLU(),
            nn.Linear(layer_dim, layer_dim),
            nn.LayerNorm(layer_dim),
            nn.ReLU(),
            nn.Linear(layer_dim, 1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        return self.net(x)


def evaluate(model, X, y):
    model.eval()
    with torch.no_grad():
        preds = model(X)
        loss = torch.nn.functional.binary_cross_entropy(preds, y)
        binary = (preds >= 0.5).float()

        accuracy = float((binary == y).float().mean().cpu().item())

        positives = y.flatten() == 1
        if positives.any():
            recall = float(((binary.flatten()[positives] == 1).float().mean()).cpu().item())
        else:
            recall = 0.0

    return {
        "loss": float(loss.cpu().item()),
        "accuracy": accuracy,
        "recall": recall,
    }


def train_model(X, y, layer_dim: int, batch_size: int, epochs: int, seed: int, val_ratio: float):
    X_train, X_val, y_train, y_val = split_train_val(X, y, seed=seed, val_ratio=val_ratio)

    train_loader = torch.utils.data.DataLoader(
        torch.utils.data.TensorDataset(torch.from_numpy(X_train), torch.from_numpy(y_train)),
        batch_size=batch_size,
        shuffle=True,
    )

    model = SimpleWakeWordNet(X.shape[1], X.shape[2], layer_dim)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

    best_state = None
    best_val_loss = float("inf")
    history = []

    X_val_t = torch.from_numpy(X_val)
    y_val_t = torch.from_numpy(y_val)

    for epoch in range(1, epochs + 1):
        model.train()
        running_loss = 0.0

        for xb, yb in train_loader:
            weights = torch.ones(yb.shape[0], 1)
            weights[yb == 1] = 0.2

            optimizer.zero_grad()
            preds = model(xb)
            loss = torch.nn.functional.binary_cross_entropy(preds, yb, weight=weights)
            loss.backward()
            optimizer.step()

            running_loss += float(loss.detach().cpu().item())

        metrics = evaluate(model, X_val_t, y_val_t)
        metrics["epoch"] = epoch
        metrics["train_loss_mean"] = running_loss / max(1, len(train_loader))
        history.append(metrics)

        print(
            f"Epoch {epoch:02d} | "
            f"train_loss={metrics['train_loss_mean']:.4f} | "
            f"val_loss={metrics['loss']:.4f} | "
            f"val_acc={metrics['accuracy']:.4f} | "
            f"val_recall={metrics['recall']:.4f}"
        )

        if metrics["loss"] < best_val_loss:
            best_val_loss = metrics["loss"]
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    if best_state is not None:
        model.load_state_dict(best_state)

    return model, {"history": history, "best_val_loss": best_val_loss}


def export_onnx(model, example_shape, output_path: Path):
    model.eval()
    dummy_input = torch.zeros((1, example_shape[0], example_shape[1]), dtype=torch.float32)

    torch.onnx.export(
        model,
        args=dummy_input,
        f=str(output_path),
        input_names=["input"],
        output_names=["score"],
        opset_version=13,
    )

## Execute training

In [ ]:
set_seed(seed)

positive_dir = positive_dir.expanduser().resolve()
negative_dir = negative_dir.expanduser().resolve()
output_dir = output_dir.expanduser().resolve()

ensure_dir_exists(positive_dir, "Positive directory")
ensure_dir_exists(negative_dir, "Negative directory")
ensure_wavs_exist(positive_dir, "positive directory")
ensure_wavs_exist(negative_dir, "negative directory")

output_dir.mkdir(parents=True, exist_ok=True)

print("Scanning and filtering audio files...")
positive_paths, positive_durations = filter_paths(
    positive_dir,
    min_positive_secs,
    max_positive_secs,
)
negative_paths, negative_durations = filter_paths(
    negative_dir,
    min_negative_secs,
    max_negative_secs,
)

if len(positive_paths) < 5:
    raise ValueError("Need at least 5 positive clips after filtering.")

if len(negative_paths) < 5:
    raise ValueError("Need at least 5 negative clips after filtering.")

print(f"Positive clips kept: {len(positive_paths)}")
print(f"Negative clips kept: {len(negative_paths)}")
print(f"Approx negative hours: {sum(negative_durations) / 3600:.2f}")

# %%
feature_extractor = openwakeword.utils.AudioFeatures()

negative_feature_path = output_dir / "negative_features.npy"
positive_feature_path = output_dir / f"{model_name}_positive_features.npy"

negative_features = build_negative_features(
    feature_extractor=feature_extractor,
    negative_paths=negative_paths,
    negative_durations=negative_durations,
    output_path=negative_feature_path,
    clip_seconds=clip_seconds,
    file_batch_size=feature_file_batch_size,
    embed_batch_size=feature_embed_batch_size,
    ncpu=feature_ncpu,
)

positive_features = build_positive_features(
    feature_extractor=feature_extractor,
    positive_paths=positive_paths,
    positive_durations=positive_durations,
    negative_paths=negative_paths,
    output_path=positive_feature_path,
    clip_seconds=clip_seconds,
    mix_batch_size=8,
    embed_batch_size=min(feature_embed_batch_size, 256),
    snr_low=snr_low,
    snr_high=snr_high,
)

print("Negative feature shape:", negative_features.shape)
print("Positive feature shape:", positive_features.shape)

# %%
X = np.vstack((negative_features, positive_features)).astype(np.float32)
y = np.array(
    [0] * len(negative_features) + [1] * len(positive_features),
    dtype=np.float32
)[:, None]

print(f"Training examples: {len(X)}")
print(f"  Negative: {len(negative_features)}")
print(f"  Positive: {len(positive_features)}")

# %%
model, metrics = train_model(
    X=X,
    y=y,
    layer_dim=layer_dim,
    batch_size=train_batch_size,
    epochs=epochs,
    seed=seed,
    val_ratio=validation_ratio,
)

# %%
onnx_path = output_dir / f"{model_name}.onnx"
torch_path = output_dir / f"{model_name}.pt"
summary_path = output_dir / "training_summary.json"

export_onnx(model, example_shape=(X.shape[1], X.shape[2]), output_path=onnx_path)
torch.save(model.state_dict(), torch_path)

summary = {
    "model_name": model_name,
    "positive_dir": str(positive_dir),
    "negative_dir": str(negative_dir),
    "positive_count": len(positive_paths),
    "negative_count": len(negative_paths),
    "negative_hours": sum(negative_durations) / 3600.0,
    "onnx_path": str(onnx_path),
    "torch_path": str(torch_path),
    "metrics": metrics,
    "input_shape": [int(X.shape[1]), int(X.shape[2])],
}

with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print("\nDone.")
print("ONNX model:", onnx_path)
print("PyTorch weights:", torch_path)
print("Summary:", summary_path)